In [ ]:
# ==================================================================================
# Kaggle T4×2 환경 설치 스크립트 (개선 버전)
# - 버전 고정으로 재현성 보장
# - GPU/CUDA 확인
# - 설치 실패 시 재시도
# - Kaggle 기본 패키지는 스킵
# ==================================================================================

import subprocess
import sys
import os


def run_cmd(cmd, retries=2, shell=False):
    """명령 실행 (재시도 지원)"""
    for attempt in range(retries):
        try:
            if shell:
                subprocess.check_call(cmd, shell=True, stderr=subprocess.DEVNULL)
            else:
                subprocess.check_call(cmd, stderr=subprocess.DEVNULL)
            return True
        except subprocess.CalledProcessError as e:
            if attempt < retries - 1:
                print(f"  ⚠️ 재시도 중... ({attempt + 1}/{retries})")
            else:
                print(f"  ❌ 실패: {e}")
                return False
    return False


def pip_install(package, version=None, quiet=True):
    """pip 설치"""
    pkg_str = f"{package}=={version}" if version else package
    cmd = [sys.executable, "-m", "pip", "install"]
    if quiet:
        cmd.append("--quiet")
    cmd.append(pkg_str)
    return run_cmd(cmd)


def check_import(import_name):
    """import 가능 여부 확인"""
    try:
        __import__(import_name)
        return True
    except ImportError:
        return False


def check_gpu():
    """GPU 및 CUDA 확인"""
    print("\n" + "=" * 70)
    print("🔍 GPU 환경 확인")
    print("=" * 70)

    try:
        import torch
        if torch.cuda.is_available():
            gpu_count = torch.cuda.device_count()
            print(f"✅ CUDA 사용 가능")
            print(f"   PyTorch: {torch.__version__}")
            print(f"   CUDA: {torch.version.cuda}")
            print(f"   GPU 수: {gpu_count}")
            for i in range(gpu_count):
                props = torch.cuda.get_device_properties(i)
                mem_gb = props.total_memory / 1024 ** 3
                print(f"   GPU {i}: {props.name} ({mem_gb:.1f}GB)")
            return gpu_count
        else:
            print("⚠️ CUDA 사용 불가 - CPU 모드로 실행됩니다")
            return 0
    except ImportError:
        print("❌ PyTorch 없음")
        return 0


def install_core_packages():
    """핵심 패키지 설치"""
    print("\n" + "=" * 70)
    print("📦 핵심 패키지 설치")
    print("=" * 70)

    # 버전 고정 패키지 목록 (Kaggle 2025.04 기준 테스트됨)
    packages = [
        # (패키지명, 버전, import명)
        ("faster-whisper", "1.0.3", "faster_whisper"),
        ("noisereduce", "3.0.2", "noisereduce"),
        ("jiwer", "3.0.4", "jiwer"),
        ("speechbrain", "1.0.1", "speechbrain"),
    ]

    # Kaggle에 이미 있는 건 스킵
    skip_packages = {"scipy", "librosa", "soundfile", "tqdm", "matplotlib", "seaborn", "pandas", "numpy"}

    results = []
    for pkg, version, import_name in packages:
        if pkg in skip_packages:
            print(f"  ⏭️ {pkg}: Kaggle 기본 패키지 (스킵)")
            results.append((pkg, "skipped"))
            continue

        if check_import(import_name):
            print(f"  ✅ {pkg}: 이미 설치됨")
            results.append((pkg, "exists"))
        else:
            print(f"  📥 {pkg}=={version} 설치 중...")
            if pip_install(pkg, version):
                print(f"  ✅ {pkg}: 설치 완료")
                results.append((pkg, "installed"))
            else:
                print(f"  ❌ {pkg}: 설치 실패")
                results.append((pkg, "failed"))

    return results


def install_mecab():
    """Mecab 설치 (한국어 형태소 분석)"""
    print("\n" + "=" * 70)
    print("🇰🇷 Mecab 설치 (한국어 형태소 분석)")
    print("=" * 70)

    # 이미 설치되어 있는지 확인
    try:
        from konlpy.tag import Mecab
        m = Mecab()
        m.morphs("테스트")
        print("  ✅ Mecab: 이미 설치됨")
        return True
    except:
        pass

    print("  📥 apt 패키지 설치 중...")

    # apt 업데이트
    if not run_cmd("apt-get update -qq", shell=True):
        print("  ⚠️ apt update 실패 - 계속 진행")

    # mecab 설치
    apt_packages = ["mecab", "libmecab-dev", "mecab-ipadic-utf8"]
    for pkg in apt_packages:
        run_cmd(f"apt-get install -y -qq {pkg}", shell=True)

    # mecab-ko-dic (한국어 사전)
    print("  📥 한국어 사전 설치 중...")
    run_cmd(
        "bash -c 'cd /tmp && git clone --depth 1 https://github.com/SOMJANG/Mecab-ko-for-Google-Colab.git && cd Mecab-ko-for-Google-Colab && bash install_mecab-ko_on_colab_light_220429.sh'",
        shell=True)

    # Python 패키지
    print("  📥 Python 패키지 설치 중...")
    pip_install("mecab-python3")
    pip_install("konlpy")

    # 확인
    try:
        from konlpy.tag import Mecab
        m = Mecab()
        result = m.morphs("설치 테스트")
        print(f"  ✅ Mecab 설치 완료: {result}")
        return True
    except Exception as e:
        print(f"  ⚠️ Mecab 설치 실패: {e}")
        print("  💡 WER 계산은 가능하지만 형태소 분석 기반 평가는 불가")
        return False


def verify_installation():
    """설치 검증"""
    print("\n" + "=" * 70)
    print("🔬 설치 검증")
    print("=" * 70)

    checks = [
        ("faster_whisper", "Whisper STT"),
        ("noisereduce", "Noise Reduction"),
        ("speechbrain", "SpeechBrain (Sepformer)"),
        ("jiwer", "WER 계산"),
        ("librosa", "오디오 처리"),
        ("torch", "PyTorch"),
    ]

    all_ok = True
    for module, desc in checks:
        if check_import(module):
            print(f"  ✅ {desc}: OK")
        else:
            print(f"  ❌ {desc}: 실패")
            all_ok = False

    return all_ok


def main():
    print("=" * 70)
    print("🚀 Kaggle T4×2 STT 파이프라인 환경 설정")
    print("=" * 70)

    # 1. GPU 확인
    gpu_count = check_gpu()

    # 2. 핵심 패키지 설치
    results = install_core_packages()

    # 3. Mecab 설치 (선택적 - 실패해도 계속)
    mecab_ok = install_mecab()

    # 4. 검증
    all_ok = verify_installation()

    # 5. 결과 요약
    print("\n" + "=" * 70)
    print("📊 설치 결과 요약")
    print("=" * 70)

    installed = sum(1 for _, status in results if status == "installed")
    failed = sum(1 for _, status in results if status == "failed")

    print(f"  GPU: {gpu_count}개 {'✅' if gpu_count > 0 else '⚠️ CPU 모드'}")
    print(f"  패키지: 신규 설치 {installed}개, 실패 {failed}개")
    print(f"  Mecab: {'✅' if mecab_ok else '⚠️ 미설치 (WER 계산은 가능)'}")

    if all_ok:
        print("\n" + "=" * 70)
        print("✨ 모든 설치 완료! 이제 메인 코드를 실행하세요.")
        print("=" * 70)
    else:
        print("\n" + "=" * 70)
        print("⚠️ 일부 설치 실패 - 위 로그를 확인하세요.")
        print("=" * 70)


if __name__ == "__main__":
    main()

In [1]:
# ==================================================================================
# 한국어 STT 통합 파이프라인 (V6 - 초보수적 SNR 기반 스킵)
# - V5 기반 + SNR ≥ 6dB에서 노이즈 제거 스킵 (초보수적)
# - V5 결과: SNR 7-9dB 영역에서 부분 스킵 시 불균형 발생
# - "Do No Harm" 원칙 최대화: SNR < 6dB에서만 처리
# - 논문 기여: 최적 SNR 임계값 탐색 (6dB vs 8dB vs 15dB)
# ==================================================================================

import os
import gc
import re
import time
import json
import glob
import torch
import shutil
import tempfile
import logging
import numpy as np
import pandas as pd
import soundfile as sf
import librosa
import librosa.display
import jiwer
import matplotlib.pyplot as plt
import matplotlib

matplotlib.use('Agg')

from tqdm import tqdm
from datetime import datetime
from pathlib import Path
from concurrent.futures import ThreadPoolExecutor, as_completed
from contextlib import contextmanager
from dataclasses import dataclass, field
from typing import Optional, List, Dict, Tuple, Any
from difflib import SequenceMatcher
from scipy import signal
from scipy.stats import ttest_rel, wilcoxon, sem
from scipy import stats as sp_stats
from queue import Queue
from threading import Thread
import warnings

warnings.filterwarnings("ignore")

plt.rcParams['font.family'] = ['DejaVu Sans', 'sans-serif']
plt.rcParams['axes.unicode_minus'] = False


# ==================================================================================
# [1] 로깅 설정
# ==================================================================================
def setup_logging(output_dir: str) -> logging.Logger:
    """체계화된 로깅 설정"""
    logger = logging.getLogger("STT_Pipeline_V6")
    logger.setLevel(logging.DEBUG)

    logger.handlers.clear()

    os.makedirs(output_dir, exist_ok=True)
    file_handler = logging.FileHandler(
        f"{output_dir}/pipeline_v6_{datetime.now().strftime('%Y%m%d_%H%M%S')}.log",
        encoding='utf-8'
    )
    file_handler.setLevel(logging.DEBUG)
    file_formatter = logging.Formatter(
        '%(asctime)s [%(levelname)s] %(name)s: %(message)s',
        datefmt='%Y-%m-%d %H:%M:%S'
    )
    file_handler.setFormatter(file_formatter)

    console_handler = logging.StreamHandler()
    console_handler.setLevel(logging.INFO)
    console_formatter = logging.Formatter('[%(levelname)s] %(message)s')
    console_handler.setFormatter(console_formatter)

    logger.addHandler(file_handler)
    logger.addHandler(console_handler)

    return logger


# ==================================================================================
# [2] 설정 (V6 - 초보수적 스킵 임계값)
# ==================================================================================
@dataclass
class Config:
    """파이프라인 설정"""
    # GPU 할당
    WHISPER_DEVICE_0: str = "cuda:0"
    WHISPER_DEVICE_1: str = "cuda:1"

    # 오디오 설정
    SAMPLE_RATE: int = 16000
    CHUNK_DURATION: int = 30
    OVERLAP_SEC: float = 3.0

    # V6 핵심: SNR 기반 스킵 임계값 (초보수적)
    SNR_SKIP_THRESHOLD: float = 6.0  # V5: 8.0 → V6: 6.0 (SNR < 6dB에서만 처리)

    # 노이즈 판단 임계값 (SNR dB 기준) - V6에서는 단순화
    SNR_CLEAN_THRESHOLD: float = 18.0
    SNR_MODERATE_THRESHOLD: float = 12.0
    SNR_NOISY_THRESHOLD: float = 4.0  # V6: 6.0 → 4.0 (더 엄격한 다단계 처리)

    # 노이즈 제거 강도 설정
    DENOISE_STRENGTH_CLEAN: float = 0.2
    DENOISE_STRENGTH_MODERATE: float = 0.4
    DENOISE_STRENGTH_NOISY: float = 0.6

    # 스펙트럼 평탄도 임계값
    SPECTRAL_FLATNESS_THRESHOLD: float = 0.15

    # 청크 병합 설정
    MERGE_SIMILARITY_THRESHOLD: float = 0.65
    MERGE_MAX_OVERLAP_WORDS: int = 15

    # SRT 파일 매칭 규칙
    SRT_SUFFIX_MAPPINGS: Dict[str, str] = field(default_factory=lambda: {
        "_NV": "_VN",
        "_VN": "_NV",
        "_noise": "_clean",
        "_noisy": "_clean",
    })

    # 병렬 처리
    ENABLE_PARALLEL: bool = True
    MAX_WORKERS: int = 2

    # 출력 디렉토리
    OUTPUT_DIR: str = "/kaggle/working/results_v6"

    def __post_init__(self):
        """설정 검증"""
        if torch.cuda.is_available():
            gpu_count = torch.cuda.device_count()
            logger_temp = logging.getLogger("Config")
            logger_temp.info(f"GPU {gpu_count}개 감지됨")

            if gpu_count < 2:
                logger_temp.warning("GPU 1개만 감지 - 단일 GPU 모드")
                self.WHISPER_DEVICE_1 = "cuda:0"
                self.ENABLE_PARALLEL = False
        else:
            self.WHISPER_DEVICE_0 = "cpu"
            self.WHISPER_DEVICE_1 = "cpu"
            self.ENABLE_PARALLEL = False

        assert 0 < self.OVERLAP_SEC < self.CHUNK_DURATION
        assert 0 < self.MERGE_SIMILARITY_THRESHOLD < 1.0
        assert self.SNR_MODERATE_THRESHOLD < self.SNR_CLEAN_THRESHOLD

        os.makedirs(self.OUTPUT_DIR, exist_ok=True)
        os.makedirs(f"{self.OUTPUT_DIR}/images", exist_ok=True)


# 전역 설정 및 로거
config = Config()
logger = setup_logging(config.OUTPUT_DIR)


# ==================================================================================
# [3] 유틸리티 함수
# ==================================================================================
def clear_gpu_memory():
    """GPU 메모리 정리"""
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        torch.cuda.synchronize()


def normalize_text(text: str) -> str:
    """텍스트 정규화 (한국어 특화)"""
    if not text:
        return ""

    text = text.lower()
    text = re.sub(r'\([^)]*\)', '', text)
    text = re.sub(r'\[[^\]]*\]', '', text)
    text = re.sub(r'[.,!?;:"\'-]', '', text)
    text = re.sub(r'\s+', ' ', text)

    return text.strip()


def calculate_wer(reference: str, hypothesis: str) -> float:
    """WER 계산 (정규화 포함)"""
    ref_norm = normalize_text(reference)
    hyp_norm = normalize_text(hypothesis)

    if not ref_norm:
        return 0.0 if not hyp_norm else 100.0

    try:
        wer = jiwer.wer(ref_norm, hyp_norm)
        return min(wer * 100, 100.0)
    except Exception:
        return 100.0


def calculate_cer(reference: str, hypothesis: str) -> float:
    """CER 계산 (문자 단위)"""
    ref_norm = normalize_text(reference)
    hyp_norm = normalize_text(hypothesis)

    if not ref_norm:
        return 0.0 if not hyp_norm else 100.0

    try:
        cer = jiwer.cer(ref_norm, hyp_norm)
        return min(cer * 100, 100.0)
    except Exception:
        return 100.0


# ==================================================================================
# [4] SRT 파서 (V3에서 가져옴)
# ==================================================================================
class SRTParser:
    """SRT 자막 파일 파서"""

    @staticmethod
    def parse_srt(srt_path: str) -> List[Dict]:
        """SRT 파일 파싱"""
        segments = []

        encodings = ['utf-8', 'cp949', 'euc-kr', 'utf-8-sig']
        content = None

        for encoding in encodings:
            try:
                with open(srt_path, 'r', encoding=encoding) as f:
                    content = f.read()
                break
            except (UnicodeDecodeError, UnicodeError):
                continue

        if content is None:
            logger.warning(f"SRT 파일 인코딩 실패: {srt_path}")
            return []

        blocks = re.split(r'\n\s*\n', content.strip())

        for block in blocks:
            lines = block.strip().split('\n')
            if len(lines) < 3:
                continue

            time_match = re.search(
                r'(\d{2}):(\d{2}):(\d{2})[,.](\d{3})\s*-->\s*(\d{2}):(\d{2}):(\d{2})[,.](\d{3})',
                lines[1]
            )

            if time_match:
                start_time = (
                        int(time_match.group(1)) * 3600 +
                        int(time_match.group(2)) * 60 +
                        int(time_match.group(3)) +
                        int(time_match.group(4)) / 1000
                )
                end_time = (
                        int(time_match.group(5)) * 3600 +
                        int(time_match.group(6)) * 60 +
                        int(time_match.group(7)) +
                        int(time_match.group(8)) / 1000
                )

                text = ' '.join(lines[2:]).strip()
                text = re.sub(r'<[^>]+>', '', text)
                text = re.sub(r'\{[^}]+\}', '', text)

                segments.append({
                    'start': start_time,
                    'end': end_time,
                    'text': text
                })

        return segments

    @staticmethod
    def get_chunk_gt(segments: List[Dict], chunk_start: float, chunk_end: float,
                     overlap_threshold: float = 0.5) -> str:
        """청크 시간 범위에 해당하는 GT 텍스트 추출"""
        matching_texts = []

        for seg in segments:
            seg_start = seg['start']
            seg_end = seg['end']
            seg_duration = seg_end - seg_start

            if seg_duration <= 0:
                continue

            overlap_start = max(seg_start, chunk_start)
            overlap_end = min(seg_end, chunk_end)
            overlap_duration = max(0, overlap_end - overlap_start)

            overlap_ratio = overlap_duration / seg_duration

            if overlap_ratio >= overlap_threshold:
                matching_texts.append(seg['text'])

        return ' '.join(matching_texts)

    @staticmethod
    def find_srt_file(audio_path: str) -> Optional[str]:
        """오디오 파일에 대응하는 SRT 파일 찾기"""
        audio_dir = os.path.dirname(audio_path)
        audio_name = os.path.splitext(os.path.basename(audio_path))[0]

        direct_srt = os.path.join(audio_dir, f"{audio_name}.srt")
        if os.path.exists(direct_srt):
            return direct_srt

        for suffix, replacement in config.SRT_SUFFIX_MAPPINGS.items():
            if audio_name.endswith(suffix):
                srt_name = audio_name[:-len(suffix)] + replacement + ".srt"
                srt_path = os.path.join(audio_dir, srt_name)
                if os.path.exists(srt_path):
                    return srt_path

        for srt_file in glob.glob(os.path.join(audio_dir, "*.srt")):
            srt_name = os.path.splitext(os.path.basename(srt_file))[0]
            if audio_name in srt_name or srt_name in audio_name:
                return srt_file

        return None


# ==================================================================================
# [5] 청크 매니저
# ==================================================================================
class ChunkManager:
    """오디오 청크 관리"""

    def __init__(self, audio_path: str, srt_segments: List[Dict] = None):
        self.audio_path = audio_path
        self.audio, self.sr = sf.read(audio_path)
        if len(self.audio.shape) > 1:
            self.audio = self.audio.mean(axis=1)
        self.duration = len(self.audio) / self.sr
        self.chunks = self._create_chunks()
        self.srt_segments = srt_segments or []

        if self.srt_segments:
            self._assign_chunk_gt()

    def _create_chunks(self) -> List[Dict]:
        """오버랩 청크 생성"""
        chunks = []
        chunk_samples = int(config.CHUNK_DURATION * self.sr)
        overlap_samples = int(config.OVERLAP_SEC * self.sr)
        hop_samples = chunk_samples - overlap_samples

        start = 0
        idx = 0

        while start < len(self.audio):
            end = min(start + chunk_samples, len(self.audio))

            chunks.append({
                "idx": idx,
                "start_sample": start,
                "end_sample": end,
                "start_sec": start / self.sr,
                "end_sec": end / self.sr,
                "audio": self.audio[start:end].copy(),
                "ground_truth": ""
            })

            if end >= len(self.audio):
                break

            start += hop_samples
            idx += 1

        return chunks

    def _assign_chunk_gt(self):
        """각 청크에 SRT 기반 GT 할당"""
        for chunk in self.chunks:
            chunk_gt = SRTParser.get_chunk_gt(
                self.srt_segments,
                chunk["start_sec"],
                chunk["end_sec"]
            )
            chunk["ground_truth"] = chunk_gt

    def get_full_gt(self) -> str:
        """전체 파일 GT 반환"""
        if not self.srt_segments:
            return ""
        return ' '.join([seg['text'] for seg in self.srt_segments])

    def save_chunk(self, chunk_idx: int, suffix: str = "") -> str:
        """청크를 임시 파일로 저장"""
        chunk = self.chunks[chunk_idx]

        base_name = os.path.splitext(os.path.basename(self.audio_path))[0]
        temp_path = f"/tmp/{base_name}_chunk{chunk_idx}{suffix}.wav"

        sf.write(temp_path, chunk["audio"], self.sr)
        return temp_path


# ==================================================================================
# [6] 노이즈 분석기
# ==================================================================================
class NoiseAnalyzer:
    """노이즈 레벨 추정"""

    @staticmethod
    def estimate_snr(audio: np.ndarray, sr: int) -> float:
        """SNR 추정 (VAD 기반)"""
        frame_length = int(0.025 * sr)
        hop_length = int(0.010 * sr)

        frames = librosa.util.frame(audio, frame_length=frame_length, hop_length=hop_length)
        energy = np.sum(frames ** 2, axis=0)

        threshold = np.percentile(energy, 30)
        speech_frames = energy > threshold
        noise_frames = ~speech_frames

        if np.sum(speech_frames) == 0 or np.sum(noise_frames) == 0:
            return 20.0

        speech_energy = np.mean(energy[speech_frames])
        noise_energy = np.mean(energy[noise_frames]) + 1e-10

        snr_db = 10 * np.log10(speech_energy / noise_energy)
        return np.clip(snr_db, -10, 40)

    @staticmethod
    def estimate_spectral_flatness(audio: np.ndarray, sr: int) -> float:
        """스펙트럼 평탄도 추정"""
        S = np.abs(librosa.stft(audio, n_fft=2048, hop_length=512))

        geometric_mean = np.exp(np.mean(np.log(S + 1e-10), axis=0))
        arithmetic_mean = np.mean(S, axis=0)

        flatness = geometric_mean / (arithmetic_mean + 1e-10)
        return float(np.mean(flatness))

    @staticmethod
    def detect_noise_type(audio: np.ndarray, sr: int) -> str:
        """노이즈 유형 감지"""
        frame_length = int(0.025 * sr)
        hop_length = int(0.010 * sr)

        frames = librosa.util.frame(audio, frame_length=frame_length, hop_length=hop_length)
        energy = np.sum(frames ** 2, axis=0)

        if len(energy) < 2:
            return "unknown"

        energy_std = np.std(energy)
        energy_mean = np.mean(energy)
        cv = energy_std / (energy_mean + 1e-10)

        spec = np.abs(librosa.stft(audio, n_fft=2048, hop_length=512))
        spec_diff = np.diff(spec, axis=1)
        spec_change = np.mean(np.abs(spec_diff))

        if cv < 0.5 and spec_change < 0.1:
            return "stationary"
        elif cv > 1.5:
            return "impulsive"
        else:
            return "non-stationary"

    @staticmethod
    def estimate_noise_level(audio: np.ndarray, sr: int) -> Tuple[float, str, Dict]:
        """종합 노이즈 레벨 추정"""
        snr_db = NoiseAnalyzer.estimate_snr(audio, sr)
        spectral_flatness = NoiseAnalyzer.estimate_spectral_flatness(audio, sr)
        noise_type = NoiseAnalyzer.detect_noise_type(audio, sr)

        details = {
            "snr_db": snr_db,
            "spectral_flatness": spectral_flatness,
            "noise_type": noise_type
        }

        if snr_db > config.SNR_CLEAN_THRESHOLD:
            return 0.2, "clean", details
        elif snr_db > config.SNR_MODERATE_THRESHOLD:
            return 0.5, "moderate", details
        else:
            return 0.8, "noisy", details


# ==================================================================================
# [7] Whisper 모델
# ==================================================================================
class WhisperModel:
    """Faster-Whisper 래퍼"""

    def __init__(self, device: str = None):
        self._model = None
        self._device = device or config.WHISPER_DEVICE_0

    def _load(self):
        if self._model is None:
            logger.info(f"Whisper large-v3 로딩 → {self._device}")
            from faster_whisper import WhisperModel as FW

            device = "cuda" if "cuda" in self._device else "cpu"
            device_index = int(self._device.split(":")[-1]) if "cuda" in self._device else 0

            self._model = FW(
                "large-v3",
                device=device,
                device_index=device_index,
                compute_type="float16" if device == "cuda" else "int8",
                cpu_threads=4,
                num_workers=2
            )
            logger.info(f"Whisper 로딩 완료 ({self._device})")

    def transcribe(self, audio_path: str, language: str = "ko") -> Tuple[str, Dict]:
        self._load()

        try:
            segments, info = self._model.transcribe(
                audio_path,
                language=language,
                beam_size=5,
                temperature=[0.0, 0.2, 0.4, 0.6],
                compression_ratio_threshold=2.4,
                log_prob_threshold=-1.0,
                no_speech_threshold=0.6,
                vad_filter=True,
                vad_parameters={
                    "threshold": 0.5,
                    "min_speech_duration_ms": 250,
                    "min_silence_duration_ms": 100,
                    "speech_pad_ms": 30,
                },
                condition_on_previous_text=False,
                word_timestamps=False,
            )

            texts = []
            total_prob = 0.0
            count = 0

            for seg in segments:
                texts.append(seg.text.strip())
                if hasattr(seg, 'avg_logprob'):
                    total_prob += seg.avg_logprob
                    count += 1

            avg_confidence = np.exp(total_prob / count) if count > 0 else 0.0

            return " ".join(texts), {
                "language": info.language,
                "language_probability": info.language_probability,
                "avg_confidence": avg_confidence,
                "duration": info.duration
            }
        except Exception as e:
            logger.error(f"Whisper 오류: {e}")
            return "", {"error": str(e)}

    def unload(self):
        if self._model is not None:
            del self._model
            self._model = None
            clear_gpu_memory()
            logger.info("Whisper 모델 언로드됨")


# ==================================================================================
# [8] 디노이저 V6 (초보수적 SNR 기반 스킵)
# ==================================================================================
class DenoiserV6:
    """V6: 초보수적 SNR 기반 노이즈 제거 (SNR >= 6dB 스킵)"""

    def __init__(self):
        self._nr = None
        self.skip_count = 0
        self.process_count = 0

    def _load_noisereduce(self):
        if self._nr is None:
            import noisereduce as nr
            self._nr = nr
            logger.info("noisereduce 로딩 완료")

    def denoise(self, audio_path: str) -> Tuple[str, str, Dict]:
        """
        V6 핵심: 초보수적 SNR 기반 노이즈 제거
        - SNR >= 6dB: 스킵 (원본 그대로) - 가장 보수적
        - SNR 4-6dB: 가벼운 처리
        - SNR < 4dB: 다단계 처리
        """
        self._load_noisereduce()

        audio, sr = sf.read(audio_path)
        if len(audio.shape) > 1:
            audio = audio.mean(axis=1)

        # 노이즈 레벨 분석
        _, category, analysis = NoiseAnalyzer.estimate_noise_level(audio, sr)
        snr_db = analysis["snr_db"]

        # V6 핵심: SNR >= 6dB 스킵 (초보수적)
        if snr_db >= config.SNR_SKIP_THRESHOLD:
            # SNR >= 6dB - 스킵 (원본 그대로 사용)
            self.skip_count += 1
            output_path = audio_path.replace(".wav", "_denoised.wav")
            sf.write(output_path, audio, sr)  # 원본 그대로 저장
            analysis["skipped"] = True
            analysis["skip_reason"] = f"SNR {snr_db:.1f}dB >= {config.SNR_SKIP_THRESHOLD}dB"
            return output_path, "skipped", analysis

        # 노이즈 제거 수행 (SNR < 6dB에서만)
        self.process_count += 1
        analysis["skipped"] = False

        if snr_db >= config.SNR_NOISY_THRESHOLD:
            # SNR 4-6dB: 가벼운 처리
            denoised, method = self._process_light(audio, sr, analysis)
        else:
            # SNR < 4dB: 다단계 처리
            denoised, method = self._process_noisy(audio, sr, analysis)

        output_path = audio_path.replace(".wav", "_denoised.wav")
        sf.write(output_path, denoised, sr)

        return output_path, method, analysis

    def _process_light(self, audio: np.ndarray, sr: int, analysis: Dict) -> Tuple[np.ndarray, str]:
        """가벼운 처리 (SNR 4-6dB)"""
        noise_type = analysis.get("noise_type", "non-stationary")

        if noise_type == "stationary":
            denoised = self._nr.reduce_noise(
                y=audio, sr=sr,
                stationary=True,
                prop_decrease=0.3,
                n_fft=2048, hop_length=512
            )
        else:
            denoised = self._nr.reduce_noise(
                y=audio, sr=sr,
                stationary=False,
                prop_decrease=0.3,
                n_fft=2048, hop_length=512,
                time_constant_s=2.0
            )
        return denoised, f"moderate_{noise_type}"

    def _process_noisy(self, audio: np.ndarray, sr: int, analysis: Dict) -> Tuple[np.ndarray, str]:
        """다단계 처리 (SNR < 4dB)"""
        noise_type = analysis.get("noise_type", "non-stationary")
        snr_db = analysis.get("snr_db", 5.0)

        # Stage 1: Stationary 노이즈 제거
        stage1 = self._nr.reduce_noise(
            y=audio, sr=sr,
            stationary=True,
            prop_decrease=0.5,
            n_fft=2048, hop_length=512
        )

        # Stage 2: 노이즈 유형별 추가 처리
        if noise_type == "impulsive":
            stage2 = self._nr.reduce_noise(
                y=stage1, sr=sr,
                stationary=False,
                prop_decrease=0.3,
                n_fft=512, hop_length=128
            )
        elif noise_type == "non-stationary":
            stage2 = self._nr.reduce_noise(
                y=stage1, sr=sr,
                stationary=False,
                prop_decrease=0.4,
                n_fft=1024, hop_length=256,
                time_constant_s=1.5
            )
        else:
            stage2 = self._nr.reduce_noise(
                y=stage1, sr=sr,
                stationary=True,
                prop_decrease=0.3,
                n_fft=2048, hop_length=512
            )

        # Stage 3: 매우 낮은 SNR일 때만 추가 Wiener
        if snr_db < 6.0:
            stage3 = self._wiener_filter(stage2, sr, strength=0.2)
        else:
            stage3 = stage2

        return stage3, f"multi_stage_{noise_type}"

    def _wiener_filter(self, audio: np.ndarray, sr: int, strength: float = 0.3) -> np.ndarray:
        """가벼운 Wiener 필터"""
        n_fft = 2048
        hop_length = 512

        S = librosa.stft(audio, n_fft=n_fft, hop_length=hop_length)
        mag, phase = np.abs(S), np.angle(S)

        noise_frames = min(10, mag.shape[1])
        noise_mag = np.mean(mag[:, :noise_frames], axis=1, keepdims=True)

        snr_est = (mag ** 2) / (noise_mag ** 2 + 1e-10)
        gain = snr_est / (1 + snr_est)
        gain = gain * strength + (1 - strength)

        S_filtered = (mag * gain) * np.exp(1j * phase)
        return librosa.istft(S_filtered, hop_length=hop_length, length=len(audio))

    def get_stats(self) -> Dict:
        """스킵/처리 통계 반환"""
        total = self.skip_count + self.process_count
        return {
            "total_chunks": total,
            "skipped": self.skip_count,
            "processed": self.process_count,
            "skip_rate": self.skip_count / total * 100 if total > 0 else 0
        }

    def unload(self):
        self._nr = None
        logger.info(f"Denoiser V6 언로드됨 (스킵: {self.skip_count}, 처리: {self.process_count})")


# ==================================================================================
# [9] 청크 병합기
# ==================================================================================
class ChunkMerger:
    """오버랩 청크 병합"""

    @staticmethod
    def merge_chunks(chunks: List[Dict]) -> str:
        """청크 텍스트 병합"""
        if not chunks:
            return ""

        if len(chunks) == 1:
            return chunks[0].get("text", "")

        merged_text = chunks[0].get("text", "")

        for i in range(1, len(chunks)):
            current_text = chunks[i].get("text", "")
            if not current_text:
                continue

            overlap_point = ChunkMerger._find_overlap(merged_text, current_text)

            if overlap_point > 0:
                merged_text += " " + current_text[overlap_point:]
            else:
                merged_text += " " + current_text

        return merged_text.strip()

    @staticmethod
    def _find_overlap(text1: str, text2: str) -> int:
        """두 텍스트 간 겹침 지점 찾기"""
        words1 = text1.split()
        words2 = text2.split()

        if len(words1) < 3 or len(words2) < 3:
            return 0

        max_overlap = min(config.MERGE_MAX_OVERLAP_WORDS, len(words1), len(words2))

        best_overlap_start = 0
        best_similarity = 0.0

        for overlap_size in range(3, max_overlap + 1):
            end_of_text1 = " ".join(words1[-overlap_size:])
            start_of_text2 = " ".join(words2[:overlap_size])

            similarity = SequenceMatcher(None, end_of_text1, start_of_text2).ratio()

            if similarity > best_similarity and similarity >= config.MERGE_SIMILARITY_THRESHOLD:
                best_similarity = similarity
                best_overlap_start = len(" ".join(words2[:overlap_size])) + 1

        return best_overlap_start


# ==================================================================================
# [10] 메트릭 계산기
# ==================================================================================
class MetricsCalculator:
    """성능 메트릭 계산"""

    @staticmethod
    def calculate_metrics(reference: str, hypothesis: str) -> Dict[str, float]:
        """종합 메트릭 계산"""
        if not reference or not hypothesis:
            return {
                "wer": None,
                "cer": None,
                "word_accuracy": None
            }

        wer = calculate_wer(reference, hypothesis)
        cer = calculate_cer(reference, hypothesis)

        return {
            "wer": wer,
            "cer": cer,
            "word_accuracy": max(0, 100 - wer)
        }

    @staticmethod
    def calculate_improvement(baseline_wer: float, denoised_wer: float) -> float:
        """WER 개선량 계산 (양수 = 개선)"""
        if baseline_wer is None or denoised_wer is None:
            return None
        return baseline_wer - denoised_wer


# ==================================================================================
# [11] 파이프라인 V6
# ==================================================================================
class PipelineV6:
    """통합 파이프라인 V6 - 초보수적 SNR 기반 스킵"""

    def __init__(self):
        self.whisper_baseline = WhisperModel(config.WHISPER_DEVICE_0)
        self.whisper_denoised = WhisperModel(config.WHISPER_DEVICE_1)
        self.denoiser = DenoiserV6()
        self.comparisons = []

    def process_file(self, audio_path: str, srt_path: str = None) -> Dict:
        """단일 파일 처리"""
        filename = os.path.basename(audio_path)
        logger.info(f"처리 시작: {filename}")

        # SRT 로드
        srt_segments = []
        if srt_path and os.path.exists(srt_path):
            srt_segments = SRTParser.parse_srt(srt_path)
            logger.info(f"SRT 로드: {len(srt_segments)}개 세그먼트")

        # 청크 생성
        chunk_mgr = ChunkManager(audio_path, srt_segments)
        logger.info(f"청크 수: {len(chunk_mgr.chunks)}, SRT 세그먼트: {len(srt_segments)}")

        # 청크별 처리
        chunk_results = []

        if config.ENABLE_PARALLEL and torch.cuda.device_count() >= 2:
            logger.info("병렬 처리 모드 (GPU 2개)")
            chunk_results = self._process_parallel(chunk_mgr, filename)
        else:
            logger.info("순차 처리 모드")
            chunk_results = self._process_sequential(chunk_mgr, filename)

        # 결과 병합
        baseline_texts = [c["baseline_text"] for c in chunk_results]
        denoised_texts = [c["denoised_text"] for c in chunk_results]

        merged_baseline = ChunkMerger.merge_chunks([{"text": t} for t in baseline_texts])
        merged_denoised = ChunkMerger.merge_chunks([{"text": t} for t in denoised_texts])

        # 전체 파일 GT 및 메트릭 계산
        full_gt = chunk_mgr.get_full_gt()

        if full_gt:
            baseline_metrics = MetricsCalculator.calculate_metrics(full_gt, merged_baseline)
            denoised_metrics = MetricsCalculator.calculate_metrics(full_gt, merged_denoised)
            improvement = MetricsCalculator.calculate_improvement(
                baseline_metrics["wer"], denoised_metrics["wer"]
            )
        else:
            baseline_metrics = {"wer": None, "cer": None}
            denoised_metrics = {"wer": None, "cer": None}
            improvement = None

        # 스킵 통계
        skip_count = sum(1 for c in chunk_results if c.get("skipped", False))

        result = {
            "filename": filename,
            "duration": chunk_mgr.duration,
            "n_chunks": len(chunk_mgr.chunks),
            "n_skipped": skip_count,
            "skip_rate": skip_count / len(chunk_mgr.chunks) * 100 if chunk_mgr.chunks else 0,
            "baseline_text": merged_baseline,
            "denoised_text": merged_denoised,
            "ground_truth": full_gt,
            "baseline_wer": baseline_metrics["wer"],
            "baseline_cer": baseline_metrics.get("cer"),
            "denoised_wer": denoised_metrics["wer"],
            "denoised_cer": denoised_metrics.get("cer"),
            "wer_improvement": improvement,
            "chunk_results": chunk_results
        }

        if baseline_metrics["wer"] is not None:
            logger.info(f"결과 - Baseline WER: {baseline_metrics['wer']:.2f}%, "
                        f"Denoised WER: {denoised_metrics['wer']:.2f}%, "
                        f"개선: {improvement:+.2f}%p, 스킵: {skip_count}/{len(chunk_mgr.chunks)}")

        return result

    def _process_parallel(self, chunk_mgr: ChunkManager, filename: str) -> List[Dict]:
        """병렬 처리"""
        logger.info("[병렬] Baseline(GPU0) + Denoise(CPU) + Denoised(GPU1)")

        results = [None] * len(chunk_mgr.chunks)

        baseline_queue = Queue()
        denoised_queue = Queue()

        def baseline_worker():
            for i, chunk in enumerate(chunk_mgr.chunks):
                chunk_path = chunk_mgr.save_chunk(i, "_baseline")
                text, info = self.whisper_baseline.transcribe(chunk_path)
                baseline_queue.put((i, text, info))
                try:
                    os.remove(chunk_path)
                except:
                    pass

        def denoise_worker():
            for i, chunk in enumerate(chunk_mgr.chunks):
                chunk_path = chunk_mgr.save_chunk(i, "_denoise")
                denoised_path, method, analysis = self.denoiser.denoise(chunk_path)
                text, info = self.whisper_denoised.transcribe(denoised_path)
                denoised_queue.put((i, text, method, analysis, info))
                try:
                    os.remove(chunk_path)
                    os.remove(denoised_path)
                except:
                    pass

        baseline_thread = Thread(target=baseline_worker)
        denoised_thread = Thread(target=denoise_worker)

        baseline_thread.start()
        denoised_thread.start()

        baseline_results = {}
        denoised_results = {}

        baseline_thread.join()
        denoised_thread.join()

        while not baseline_queue.empty():
            i, text, info = baseline_queue.get()
            baseline_results[i] = (text, info)

        while not denoised_queue.empty():
            i, text, method, analysis, info = denoised_queue.get()
            denoised_results[i] = (text, method, analysis, info)

        for i, chunk in enumerate(chunk_mgr.chunks):
            baseline_text, baseline_info = baseline_results.get(i, ("", {}))
            denoised_text, method, analysis, denoised_info = denoised_results.get(i, ("", "", {}, {}))

            chunk_gt = chunk.get("ground_truth", "")

            if chunk_gt:
                baseline_wer = calculate_wer(chunk_gt, baseline_text)
                baseline_cer = calculate_cer(chunk_gt, baseline_text)
                denoised_wer = calculate_wer(chunk_gt, denoised_text)
                denoised_cer = calculate_cer(chunk_gt, denoised_text)
                wer_imp = baseline_wer - denoised_wer
            else:
                baseline_wer = baseline_cer = denoised_wer = denoised_cer = wer_imp = None

            result = {
                "idx": i,
                "filename": filename,
                "start_sec": chunk["start_sec"],
                "end_sec": chunk["end_sec"],
                "baseline_text": baseline_text,
                "denoised_text": denoised_text,
                "chunk_ground_truth": chunk_gt,
                "baseline_wer": baseline_wer,
                "baseline_cer": baseline_cer,
                "denoised_wer": denoised_wer,
                "denoised_cer": denoised_cer,
                "wer_improvement": wer_imp,
                "denoise_method": method,
                "snr_db": analysis.get("snr_db", 0),
                "noise_type": analysis.get("noise_type", "unknown"),
                "skipped": analysis.get("skipped", False),
                "skip_reason": analysis.get("skip_reason", "")
            }

            results[i] = result
            self.comparisons.append(result)

        return results

    def _process_sequential(self, chunk_mgr: ChunkManager, filename: str) -> List[Dict]:
        """순차 처리"""
        results = []

        for i, chunk in enumerate(chunk_mgr.chunks):
            chunk_path = chunk_mgr.save_chunk(i)

            baseline_text, _ = self.whisper_baseline.transcribe(chunk_path)
            denoised_path, method, analysis = self.denoiser.denoise(chunk_path)
            denoised_text, _ = self.whisper_baseline.transcribe(denoised_path)

            chunk_gt = chunk.get("ground_truth", "")

            if chunk_gt:
                baseline_wer = calculate_wer(chunk_gt, baseline_text)
                baseline_cer = calculate_cer(chunk_gt, baseline_text)
                denoised_wer = calculate_wer(chunk_gt, denoised_text)
                denoised_cer = calculate_cer(chunk_gt, denoised_text)
                wer_imp = baseline_wer - denoised_wer
            else:
                baseline_wer = baseline_cer = denoised_wer = denoised_cer = wer_imp = None

            result = {
                "idx": i,
                "filename": filename,
                "start_sec": chunk["start_sec"],
                "end_sec": chunk["end_sec"],
                "baseline_text": baseline_text,
                "denoised_text": denoised_text,
                "chunk_ground_truth": chunk_gt,
                "baseline_wer": baseline_wer,
                "baseline_cer": baseline_cer,
                "denoised_wer": denoised_wer,
                "denoised_cer": denoised_cer,
                "wer_improvement": wer_imp,
                "denoise_method": method,
                "snr_db": analysis.get("snr_db", 0),
                "noise_type": analysis.get("noise_type", "unknown"),
                "skipped": analysis.get("skipped", False),
                "skip_reason": analysis.get("skip_reason", "")
            }

            results.append(result)
            self.comparisons.append(result)

            try:
                os.remove(chunk_path)
                os.remove(denoised_path)
            except:
                pass

        return results

    def unload(self):
        """리소스 해제"""
        self.whisper_baseline.unload()
        self.whisper_denoised.unload()
        self.denoiser.unload()


# ==================================================================================
# [12] 리포트 생성기 V6
# ==================================================================================
class ReportGeneratorV6:
    """V6 리포트 생성"""

    @staticmethod
    def generate_html_report(results: List[dict], comparisons: List[dict],
                             output_path: str, denoiser_stats: Dict = None):
        """HTML 리포트 생성"""
        valid_results = [r for r in results if r.get("baseline_wer") is not None]
        valid_comparisons = [c for c in comparisons if c.get("baseline_wer") is not None]

        if valid_results:
            avg_baseline_wer = np.mean([r["baseline_wer"] for r in valid_results])
            avg_denoised_wer = np.mean([r["denoised_wer"] for r in valid_results])
            avg_improvement = np.mean([r["wer_improvement"] for r in valid_results])
            avg_baseline_cer = np.mean([r["baseline_cer"] for r in valid_results if r.get("baseline_cer")])
            avg_denoised_cer = np.mean([r["denoised_cer"] for r in valid_results if r.get("denoised_cer")])
        else:
            avg_baseline_wer = avg_denoised_wer = avg_improvement = 0
            avg_baseline_cer = avg_denoised_cer = 0

        # 스킵 통계
        total_skipped = sum(r.get("n_skipped", 0) for r in results)
        total_chunks = sum(r.get("n_chunks", 0) for r in results)
        skip_rate = total_skipped / total_chunks * 100 if total_chunks > 0 else 0

        html = f"""<!DOCTYPE html>
<html>
<head>
    <meta charset="UTF-8">
    <title>노이즈 제거 STT 비교 리포트 V6</title>
    <style>
        body {{ font-family: 'Segoe UI', Tahoma, Geneva, Verdana, sans-serif; margin: 20px; background: #f5f5f5; }}
        .header {{ background: linear-gradient(135deg, #1a237e 0%, #4a148c 100%); color: white; padding: 30px; border-radius: 10px; margin-bottom: 20px; }}
        .header h1 {{ margin: 0; font-size: 2em; }}
        .header .subtitle {{ opacity: 0.9; margin-top: 10px; }}
        .stats-grid {{ display: grid; grid-template-columns: repeat(auto-fit, minmax(150px, 1fr)); gap: 15px; margin-bottom: 20px; }}
        .stat-card {{ background: white; padding: 20px; border-radius: 10px; text-align: center; box-shadow: 0 2px 10px rgba(0,0,0,0.1); }}
        .stat-value {{ font-size: 2em; font-weight: bold; color: #1a237e; }}
        .stat-label {{ color: #666; font-size: 0.9em; margin-top: 5px; }}
        .skip-card {{ background: linear-gradient(135deg, #e8f5e9 0%, #c8e6c9 100%); }}
        .skip-value {{ color: #2e7d32 !important; }}
        .chunk-card {{ background: white; margin: 15px 0; padding: 20px; border-radius: 10px; box-shadow: 0 2px 10px rgba(0,0,0,0.1); }}
        .chunk-header {{ display: flex; justify-content: space-between; align-items: center; margin-bottom: 15px; padding-bottom: 15px; border-bottom: 1px solid #eee; flex-wrap: wrap; gap: 10px; }}
        .metrics-row {{ display: flex; gap: 10px; flex-wrap: wrap; }}
        .method-badge {{ background: #e3f2fd; color: #1565c0; padding: 5px 12px; border-radius: 20px; font-size: 0.85em; }}
        .skip-badge {{ background: #e8f5e9; color: #2e7d32; padding: 5px 12px; border-radius: 20px; font-size: 0.85em; font-weight: bold; }}
        .snr-badge {{ background: #fff3e0; color: #ef6c00; padding: 5px 12px; border-radius: 20px; font-size: 0.85em; }}
        .wer-badge {{ padding: 5px 12px; border-radius: 20px; font-size: 0.85em; }}
        .wer-improved {{ background: #e8f5e9; color: #2e7d32; }}
        .wer-degraded {{ background: #ffebee; color: #c62828; }}
        .wer-neutral {{ background: #f5f5f5; color: #666; }}
        .wer-skipped {{ background: #e3f2fd; color: #1565c0; }}
        .text-section {{ margin: 10px 0; }}
        .text-label {{ font-weight: 600; color: #333; margin-bottom: 5px; display: flex; align-items: center; gap: 8px; }}
        .text-content {{ background: #f8f9fa; padding: 12px; border-radius: 8px; font-size: 0.95em; line-height: 1.6; white-space: pre-wrap; }}
        .version-badge {{ background: #4a148c; color: white; padding: 3px 10px; border-radius: 12px; font-size: 0.8em; margin-left: 10px; }}
        .adaptive-badge {{ background: #00695c; color: white; padding: 3px 10px; border-radius: 12px; font-size: 0.8em; margin-left: 5px; }}
    </style>
</head>
<body>
    <div class="header">
        <h1>노이즈 제거 STT 비교 리포트 <span class="version-badge">V6</span> <span class="adaptive-badge">초보수적 스킵</span></h1>
        <div class="subtitle">생성: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')} | SNR ≥ {config.SNR_SKIP_THRESHOLD}dB → 노이즈 제거 스킵</div>
    </div>

    <div class="stats-grid">
        <div class="stat-card">
            <div class="stat-value">{len(valid_comparisons)}</div>
            <div class="stat-label">총 청크 (유효 GT)</div>
        </div>
        <div class="stat-card skip-card">
            <div class="stat-value skip-value">{total_skipped}/{total_chunks}</div>
            <div class="stat-label">스킵된 청크 ({skip_rate:.1f}%)</div>
        </div>
        <div class="stat-card">
            <div class="stat-value">{avg_baseline_wer:.1f}%</div>
            <div class="stat-label">Baseline WER</div>
        </div>
        <div class="stat-card">
            <div class="stat-value">{avg_denoised_wer:.1f}%</div>
            <div class="stat-label">Denoised WER</div>
        </div>
        <div class="stat-card">
            <div class="stat-value" style="color: {'#2e7d32' if avg_improvement > 0 else '#c62828' if avg_improvement < 0 else '#666'}">{avg_improvement:+.1f}%p</div>
            <div class="stat-label">WER 개선</div>
        </div>
        <div class="stat-card">
            <div class="stat-value">{avg_baseline_cer:.1f}%</div>
            <div class="stat-label">Baseline CER</div>
        </div>
        <div class="stat-card">
            <div class="stat-value">{avg_denoised_cer:.1f}%</div>
            <div class="stat-label">Denoised CER</div>
        </div>
    </div>

    <h2>청크별 비교 (V6 - 초보수적 스킵)</h2>
"""

        for comp in comparisons:
            method = comp.get("denoise_method", "unknown")
            noise_type = comp.get("noise_type", "unknown")
            snr = comp.get("snr_db", 0)
            skipped = comp.get("skipped", False)
            skip_reason = comp.get("skip_reason", "")

            baseline_wer = comp.get("baseline_wer")
            denoised_wer = comp.get("denoised_wer")
            baseline_cer = comp.get("baseline_cer")
            denoised_cer = comp.get("denoised_cer")
            wer_imp = comp.get("wer_improvement")

            # None 값 안전하게 처리
            baseline_wer_safe = baseline_wer if baseline_wer is not None else 0
            denoised_wer_safe = denoised_wer if denoised_wer is not None else 0
            baseline_cer_safe = baseline_cer if baseline_cer is not None else 0
            denoised_cer_safe = denoised_cer if denoised_cer is not None else 0

            if skipped:
                wer_class = "wer-skipped"
                wer_display = f"WER: {baseline_wer_safe:.1f}% (스킵됨)"
                imp_display = "스킵"
                baseline_wer_str = f"{baseline_wer_safe:.1f}%"
                denoised_wer_str = f"{denoised_wer_safe:.1f}% (=원본)"
                baseline_cer_str = f"{baseline_cer_safe:.1f}%"
                denoised_cer_str = f"{denoised_cer_safe:.1f}%"
            elif baseline_wer is not None and denoised_wer is not None:
                wer_display = f"WER: {baseline_wer_safe:.1f}%→{denoised_wer_safe:.1f}%"
                baseline_wer_str = f"{baseline_wer_safe:.1f}%"
                denoised_wer_str = f"{denoised_wer_safe:.1f}%"
                baseline_cer_str = f"{baseline_cer_safe:.1f}%"
                denoised_cer_str = f"{denoised_cer_safe:.1f}%"
                if wer_imp and wer_imp > 0.5:
                    wer_class = "wer-improved"
                    imp_display = f"+{wer_imp:.1f}%p"
                elif wer_imp and wer_imp < -0.5:
                    wer_class = "wer-degraded"
                    imp_display = f"{wer_imp:.1f}%p"
                else:
                    wer_class = "wer-neutral"
                    imp_display = f"{wer_imp:.1f}%p" if wer_imp is not None else "N/A"
            else:
                wer_display = "WER: N/A (GT 없음)"
                wer_class = "wer-neutral"
                imp_display = "N/A"
                baseline_wer_str = "N/A"
                denoised_wer_str = "N/A"
                baseline_cer_str = "N/A"
                denoised_cer_str = "N/A"

            chunk_gt = comp.get("chunk_ground_truth", "") or ""
            gt_preview = chunk_gt[:200] + "..." if len(chunk_gt) > 200 else chunk_gt
            baseline_text = comp.get("baseline_text", "") or ""
            denoised_text = comp.get("denoised_text", "") or ""
            baseline_preview = baseline_text[:200]
            denoised_preview = denoised_text[:200]

            # 스킵 배지
            skip_badge = '<span class="skip-badge">⏭️ SKIPPED</span>' if skipped else ''
            method_badge = f'<span class="method-badge">{method}</span>' if not skipped else ''

            html += f"""
        <div class="chunk-card">
            <div class="chunk-header">
                <div>
                    <strong>{comp['filename']}</strong> - 청크 {comp['idx']} | 
                    {comp['start_sec']:.0f}초 - {comp['end_sec']:.0f}초
                </div>
                <div class="metrics-row">
                    {skip_badge}
                    {method_badge}
                    <span class="method-badge">{noise_type}</span>
                    <span class="snr-badge">SNR: {snr:.1f}dB</span>
                    <span class="wer-badge {wer_class}">{wer_display} ({imp_display})</span>
                </div>
            </div>

            <div class="text-section">
                <div class="text-label">
                    📝 청크 GT ({len(chunk_gt.split()) if chunk_gt else 0}단어, {len(chunk_gt)}자)
                </div>
                <div class="text-content">{gt_preview if gt_preview else '<em>GT 없음</em>'}</div>
            </div>

            <div class="text-section">
                <div class="text-label">
                    🔴 Baseline (WER: {baseline_wer_str} | CER: {baseline_cer_str})
                </div>
                <div class="text-content">{baseline_preview if baseline_preview else '<em>인식 결과 없음</em>'}</div>
            </div>

            <div class="text-section">
                <div class="text-label">
                    {'🟢' if not skipped else '⏭️'} {'Denoised' if not skipped else 'Skipped'} [{method}] (WER: {denoised_wer_str} | CER: {denoised_cer_str})
                </div>
                <div class="text-content">{denoised_preview if denoised_preview else '<em>인식 결과 없음</em>'}</div>
            </div>
        </div>
"""

        html += """
    </div>
</body>
</html>
"""

        with open(output_path, 'w', encoding='utf-8') as f:
            f.write(html)

        logger.info(f"HTML 리포트: {output_path}")

    @staticmethod
    def generate_csv(comparisons: List[dict], output_path: str):
        """CSV 저장"""
        df = pd.DataFrame(comparisons)
        df.to_csv(output_path, index=False, encoding='utf-8-sig')
        logger.info(f"CSV 데이터: {output_path}")

    @staticmethod
    def generate_statistics(results: List[dict], comparisons: List[dict],
                            output_path: str, denoiser_stats: Dict = None) -> dict:
        """통계 JSON 생성"""
        valid_results = [r for r in results if r.get("baseline_wer") is not None]
        valid_comparisons = [c for c in comparisons if c.get("baseline_wer") is not None]

        if not valid_results:
            return {}

        baseline_wers = [r["baseline_wer"] for r in valid_results]
        denoised_wers = [r["denoised_wer"] for r in valid_results]
        improvements = [r["wer_improvement"] for r in valid_results]

        # 청크별 통계
        chunk_baseline_wers = [c["baseline_wer"] for c in valid_comparisons]
        chunk_denoised_wers = [c["denoised_wer"] for c in valid_comparisons]
        chunk_improvements = [c["wer_improvement"] for c in valid_comparisons if c["wer_improvement"] is not None]

        # 스킵 통계
        skipped_chunks = [c for c in comparisons if c.get("skipped", False)]
        processed_chunks = [c for c in comparisons if not c.get("skipped", False)]

        # 스킵된 청크 vs 처리된 청크 성능 비교
        skipped_valid = [c for c in skipped_chunks if c.get("baseline_wer") is not None]
        processed_valid = [c for c in processed_chunks if c.get("baseline_wer") is not None]

        stats = {
            "version": "V6 - 초보수적 SNR 기반 스킵",
            "snr_skip_threshold": config.SNR_SKIP_THRESHOLD,
            "generated_at": datetime.now().isoformat(),
            "file_level": {
                "n_files": len(valid_results),
                "baseline_wer_mean": float(np.mean(baseline_wers)),
                "baseline_wer_std": float(np.std(baseline_wers)),
                "denoised_wer_mean": float(np.mean(denoised_wers)),
                "denoised_wer_std": float(np.std(denoised_wers)),
                "mean_improvement": float(np.mean(improvements)),
                "improved_files": sum(1 for i in improvements if i > 0),
            },
            "chunk_level": {
                "n_chunks": len(valid_comparisons),
                "baseline_wer_mean": float(np.mean(chunk_baseline_wers)) if chunk_baseline_wers else 0,
                "denoised_wer_mean": float(np.mean(chunk_denoised_wers)) if chunk_denoised_wers else 0,
                "mean_improvement": float(np.mean(chunk_improvements)) if chunk_improvements else 0,
                "improved_chunks": sum(1 for i in chunk_improvements if i > 0),
            },
            "skip_statistics": {
                "total_chunks": len(comparisons),
                "skipped_chunks": len(skipped_chunks),
                "processed_chunks": len(processed_chunks),
                "skip_rate": len(skipped_chunks) / len(comparisons) * 100 if comparisons else 0,
                "skipped_avg_snr": float(np.mean([c["snr_db"] for c in skipped_chunks])) if skipped_chunks else 0,
                "processed_avg_snr": float(np.mean([c["snr_db"] for c in processed_chunks])) if processed_chunks else 0,
            },
            "skip_vs_process_comparison": {
                "skipped_baseline_wer": float(
                    np.mean([c["baseline_wer"] for c in skipped_valid])) if skipped_valid else None,
                "skipped_denoised_wer": float(
                    np.mean([c["denoised_wer"] for c in skipped_valid])) if skipped_valid else None,
                "processed_baseline_wer": float(
                    np.mean([c["baseline_wer"] for c in processed_valid])) if processed_valid else None,
                "processed_denoised_wer": float(
                    np.mean([c["denoised_wer"] for c in processed_valid])) if processed_valid else None,
                "processed_improvement": float(np.mean([c["wer_improvement"] for c in processed_valid if c[
                    "wer_improvement"] is not None])) if processed_valid else None,
            }
        }

        # 통계 검정
        if len(baseline_wers) >= 3:
            try:
                t_stat, t_pval = ttest_rel(baseline_wers, denoised_wers)
                stats["statistical_tests"] = {
                    "paired_t_statistic": float(t_stat),
                    "paired_t_p_value": float(t_pval),
                    "significant_at_0.05": bool(t_pval < 0.05)
                }
            except Exception:
                pass

        with open(output_path, 'w', encoding='utf-8') as f:
            json.dump(stats, f, indent=2, ensure_ascii=False)

        logger.info(f"통계: {output_path}")
        return stats


# ==================================================================================
# [13] 메인 실행
# ==================================================================================
def find_audio_files(data_dir: str) -> List[Tuple[str, Optional[str]]]:
    """오디오 파일과 대응 SRT 찾기"""
    audio_files = []

    patterns = ["**/*.wav", "**/*.WAV", "**/*.mp3", "**/*.flac"]

    for pattern in patterns:
        for audio_path in glob.glob(os.path.join(data_dir, pattern), recursive=True):
            srt_path = SRTParser.find_srt_file(audio_path)
            audio_files.append((audio_path, srt_path))

    return audio_files


def main():
    """메인 실행"""
    logger.info("=" * 60)
    logger.info("한국어 STT 통합 파이프라인 V6")
    logger.info("(SNR 기반 적응적 노이즈 제거)")
    logger.info(f"스킵 임계값: SNR ≥ {config.SNR_SKIP_THRESHOLD}dB")
    logger.info("=" * 60)

    # GPU 정보
    if torch.cuda.is_available():
        for i in range(torch.cuda.device_count()):
            props = torch.cuda.get_device_properties(i)
            logger.info(f"GPU {i}: {props.name} ({props.total_memory / 1e9:.1f}GB)")

    # 데이터 찾기
    data_dirs = [
        "/kaggle/input",
        "/content/drive/MyDrive",
        "./data",
        "."
    ]

    audio_files = []
    for data_dir in data_dirs:
        if os.path.exists(data_dir):
            audio_files = find_audio_files(data_dir)
            if audio_files:
                break

    if not audio_files:
        logger.error("오디오 파일을 찾을 수 없습니다.")
        return

    logger.info(f"발견된 파일: {len(audio_files)}개")

    # SRT 매칭 확인
    srt_matched = sum(1 for _, srt in audio_files if srt is not None)
    logger.info(f"SRT 매칭: {srt_matched}/{len(audio_files)}개")

    # 파이프라인 실행
    pipeline = PipelineV6()
    results = []

    try:
        for audio_path, srt_path in audio_files:
            logger.info("=" * 60)
            result = pipeline.process_file(audio_path, srt_path)
            results.append(result)

        logger.info("=" * 60)
        logger.info("리포트 생성 중...")

        # 리포트 생성
        ReportGeneratorV6.generate_html_report(
            results,
            pipeline.comparisons,
            f"{config.OUTPUT_DIR}/comparison_report_v6.html",
            pipeline.denoiser.get_stats()
        )

        ReportGeneratorV6.generate_csv(
            pipeline.comparisons,
            f"{config.OUTPUT_DIR}/detailed_results_v6.csv"
        )

        stats = ReportGeneratorV6.generate_statistics(
            results,
            pipeline.comparisons,
            f"{config.OUTPUT_DIR}/statistics_v6.json",
            pipeline.denoiser.get_stats()
        )

        # 최종 결과 출력
        logger.info("=" * 60)
        logger.info("📊 최종 결과 (V6 - 초보수적 스킵)")
        logger.info("=" * 60)

        if stats:
            fl = stats.get("file_level", {})
            cl = stats.get("chunk_level", {})
            ss = stats.get("skip_statistics", {})

            logger.info(f"  처리 파일: {fl.get('n_files', 0)}개")
            logger.info(f"  [스킵 통계]")
            logger.info(
                f"    스킵된 청크: {ss.get('skipped_chunks', 0)}/{ss.get('total_chunks', 0)} ({ss.get('skip_rate', 0):.1f}%)")
            logger.info(f"    스킵 청크 평균 SNR: {ss.get('skipped_avg_snr', 0):.1f}dB")
            logger.info(f"    처리 청크 평균 SNR: {ss.get('processed_avg_snr', 0):.1f}dB")
            logger.info(f"  [파일 단위]")
            logger.info(
                f"    Baseline WER: {fl.get('baseline_wer_mean', 0):.2f}% ± {fl.get('baseline_wer_std', 0):.2f}")
            logger.info(
                f"    Denoised WER: {fl.get('denoised_wer_mean', 0):.2f}% ± {fl.get('denoised_wer_std', 0):.2f}")
            logger.info(f"    평균 WER 개선: {fl.get('mean_improvement', 0):+.2f}%p")
            logger.info(f"    개선된 파일: {fl.get('improved_files', 0)}/{fl.get('n_files', 0)}")
            logger.info(f"  [청크 단위]")
            logger.info(f"    유효 청크: {cl.get('n_chunks', 0)}개")
            logger.info(f"    Baseline WER: {cl.get('baseline_wer_mean', 0):.2f}%")
            logger.info(f"    Denoised WER: {cl.get('denoised_wer_mean', 0):.2f}%")
            logger.info(f"    개선된 청크: {cl.get('improved_chunks', 0)}/{cl.get('n_chunks', 0)}")

    finally:
        pipeline.unload()
        logger.info("파이프라인 V6 종료")

    print(f"\n[INFO] ✅ V6 완료!")

    # 통계 출력
    if stats:
        print(json.dumps(stats, indent=2, ensure_ascii=False))


if __name__ == "__main__":
    main()

[INFO] ======================================================================
[INFO] V7.1 파이프라인 초기화 - 보수적 앙상블
[INFO] ======================================================================
[INFO]   Demucs: OFF
[INFO]   품질 마진: 0.25
[INFO]   SNR Skip: >= 15.0dB
[INFO] 
발견된 파일: 6개
[INFO] 
[INFO] 처리 시작: 02_04_134234_220803_0156_VN.wav
[INFO] ============================================================
[INFO] SRT 로드: 21개 세그먼트



🚀 V7.1 파이프라인 - 보수적 앙상블 + 품질 기반 선택
   (Demucs OFF, 품질 마진 상향, SNR Skip 강화)
[CONFIG] GPU 2개 감지됨
  GPU 0: Tesla T4 (15.8GB)
  GPU 1: Tesla T4 (15.8GB)

[CONFIG] V7.1 보수적 설정:
  - Demucs: OFF (한국어 왜곡 방지)
  - 품질 마진: 0.25
  - SNR Skip 임계값: >= 15.0dB

📁 데이터 디렉토리: /kaggle/input
📄 발견된 파일: 6개


[INFO] 청크 수: 11, 총 길이: 299.4초


[Whisper] large-v3 로딩 중... (device: cuda:0)
[Whisper] 로딩 완료


[INFO]   청크 0: ➡️ WER 26.7%→26.7% (+0.0%p) | 선택: original_skip [SKIP] | SNR: 16.8dB
[INFO]   청크 1: ➡️ WER 70.7%→70.7% (+0.0%p) | 선택: original_skip [SKIP] | SNR: 16.2dB
[INFO]   청크 2: ➡️ WER 13.0%→13.0% (+0.0%p) | 선택: original_skip [SKIP] | SNR: 16.8dB
[INFO]   청크 3: ✅ WER 18.6%→11.6% (+7.0%p) | 선택: spectral_gate | SNR: 14.5dB
[INFO]   청크 4: ➡️ WER 40.7%→40.7% (+0.0%p) | 선택: original_skip [SKIP] | SNR: 17.6dB
[INFO]   청크 5: ⚠️ WER 7.5%→15.0% (-7.5%p) | 선택: spectral_gate | SNR: 14.0dB
[INFO]   청크 6: ➡️ WER 13.5%→13.5% (+0.0%p) | 선택: original_skip [SKIP] | SNR: 16.3dB
[INFO]   청크 7: ➡️ WER 46.7%→46.7% (+0.0%p) | 선택: original_skip [SKIP] | SNR: 18.1dB
[INFO]   청크 8: ➡️ WER 15.0%→15.0% (+0.0%p) | 선택: original_skip [SKIP] | SNR: 17.2dB
[INFO]   청크 9: ➡️ WER 44.3%→44.3% (+0.0%p) | 선택: original_skip [SKIP] | SNR: 18.6dB
[INFO]   청크 10: ➡️ WER 40.6%→40.6% (+0.0%p) | 선택: original_skip [SKIP] | SNR: 18.5dB
[INFO] 
📊 파일 결과: 02_04_134234_220803_0156_VN.wav
[INFO]   Baseline WER: 7.93%
[INFO]   Ense


📊 V7.1 파이프라인 최종 결과 (보수적 앙상블)

[설정]
  Demucs: OFF
  품질 마진: 0.25
  SNR Skip: >= 15.0dB

[청크 레벨]
  총 청크: 67개
  Baseline WER: 45.99% ± 25.18
  Ensemble WER: 45.71% ± 25.08
  평균 개선: +0.28%p
  개선/악화/유지: 5/2/60

[경로 선택 분포]
  original: 27개 (39.7%)
  original_skip: 33개 (48.5%)
  spectral_gate: 5개 (7.4%)
  noisereduce: 3개 (4.4%)
  demucs: 0개 (0.0%)

[통계 검정]
  Paired t-test: t=1.025, p=0.3092 (❌ 유의하지 않음)



[INFO] V7.1 파이프라인 종료



✅ V7.1 파이프라인 완료!


In [ ]:
"""
Kaggle Working 디렉토리 정리 스크립트
- pretrained_models는 보존 (재다운로드 방지)
- 나머지 결과 폴더들 자동 감지 후 삭제
"""

import shutil
import os
from pathlib import Path
from datetime import datetime

# ============================================================
# 설정
# ============================================================
WORKING_DIR = "/kaggle/working"

# 절대 삭제하면 안 되는 폴더 (화이트리스트)
PRESERVE_FOLDERS = {
    "pretrained_models",  # 모델 캐시 (다운로드 오래 걸림)
    "sepformer",          # Sepformer 모델
    ".cache",             # 시스템 캐시
}

# 삭제할 파일 확장자 (루트에 있는 파일들)
DELETE_EXTENSIONS = {
    ".wav", ".mp3", ".flac",  # 오디오
    ".csv", ".json",          # 데이터
    ".png", ".jpg", ".html",  # 결과물
    ".log",                   # 로그
}

# ============================================================
# 메인 로직
# ============================================================
def get_folder_size(path: str) -> int:
    """폴더 크기 (bytes)"""
    total = 0
    try:
        for entry in os.scandir(path):
            if entry.is_file():
                total += entry.stat().st_size
            elif entry.is_dir():
                total += get_folder_size(entry.path)
    except PermissionError:
        pass
    return total


def format_size(size_bytes: int) -> str:
    """바이트를 읽기 좋게 변환"""
    for unit in ['B', 'KB', 'MB', 'GB']:
        if size_bytes < 1024:
            return f"{size_bytes:.1f}{unit}"
        size_bytes /= 1024
    return f"{size_bytes:.1f}TB"


def scan_working_dir():
    """작업 디렉토리 스캔"""
    if not os.path.exists(WORKING_DIR):
        print(f"❌ 디렉토리 없음: {WORKING_DIR}")
        return [], []
    
    folders_to_delete = []
    files_to_delete = []
    
    for entry in os.scandir(WORKING_DIR):
        if entry.is_dir():
            if entry.name not in PRESERVE_FOLDERS:
                size = get_folder_size(entry.path)
                folders_to_delete.append({
                    "path": entry.path,
                    "name": entry.name,
                    "size": size
                })
        elif entry.is_file():
            ext = os.path.splitext(entry.name)[1].lower()
            if ext in DELETE_EXTENSIONS:
                files_to_delete.append({
                    "path": entry.path,
                    "name": entry.name,
                    "size": entry.stat().st_size
                })
    
    return folders_to_delete, files_to_delete


def cleanup(dry_run: bool = False):
    """
    정리 실행
    
    Args:
        dry_run: True면 실제 삭제 안 하고 미리보기만
    """
    print("=" * 60)
    print(f"🧹 Kaggle Working 디렉토리 정리")
    print(f"   시간: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
    print(f"   모드: {'미리보기 (Dry Run)' if dry_run else '실제 삭제'}")
    print("=" * 60)
    
    folders, files = scan_working_dir()
    
    if not folders and not files:
        print("\n✨ 정리할 항목이 없습니다!")
        return
    
    total_size = 0
    
    # 폴더 목록
    if folders:
        print(f"\n📁 삭제 대상 폴더 ({len(folders)}개):")
        print("-" * 50)
        for f in sorted(folders, key=lambda x: x['size'], reverse=True):
            print(f"  {'🗑️' if not dry_run else '👀'} {f['name']:30} {format_size(f['size']):>10}")
            total_size += f['size']
    
    # 파일 목록
    if files:
        print(f"\n📄 삭제 대상 파일 ({len(files)}개):")
        print("-" * 50)
        for f in sorted(files, key=lambda x: x['size'], reverse=True)[:10]:  # 상위 10개만
            print(f"  {'🗑️' if not dry_run else '👀'} {f['name']:30} {format_size(f['size']):>10}")
            total_size += f['size']
        if len(files) > 10:
            print(f"  ... 외 {len(files) - 10}개 파일")
    
    print("-" * 50)
    print(f"📊 총 정리 용량: {format_size(total_size)}")
    
    # 보존되는 폴더 표시
    preserved = [p for p in PRESERVE_FOLDERS if os.path.exists(os.path.join(WORKING_DIR, p))]
    if preserved:
        print(f"\n🛡️ 보존되는 폴더: {', '.join(preserved)}")
    
    if dry_run:
        print("\n" + "=" * 60)
        print("💡 실제 삭제하려면: cleanup(dry_run=False)")
        print("=" * 60)
        return
    
    # 실제 삭제
    print("\n🚀 삭제 진행 중...")
    deleted_count = 0
    error_count = 0
    
    for f in folders:
        try:
            shutil.rmtree(f['path'])
            deleted_count += 1
        except Exception as e:
            print(f"  ❌ 실패: {f['name']} - {e}")
            error_count += 1
    
    for f in files:
        try:
            os.remove(f['path'])
            deleted_count += 1
        except Exception as e:
            print(f"  ❌ 실패: {f['name']} - {e}")
            error_count += 1
    
    print("\n" + "=" * 60)
    print(f"✅ 완료! 삭제: {deleted_count}개, 실패: {error_count}개")
    print(f"📦 확보된 용량: {format_size(total_size)}")
    print("=" * 60)


def quick_clean():
    """빠른 정리 (확인 없이 바로 삭제)"""
    cleanup(dry_run=False)


def preview():
    """미리보기만"""
    cleanup(dry_run=True)


# ============================================================
# 실행
# ============================================================
if __name__ == "__main__":
    import sys
    
    if len(sys.argv) > 1 and sys.argv[1] == "--force":
        quick_clean()
    else:
        # 기본: 미리보기 먼저 보여주고 확인
        preview()
        
        print("\n")
        confirm = input("정말 삭제하시겠습니까? (y/N): ").strip().lower()
        if confirm == 'y':
            cleanup(dry_run=False)
        else:
            print("취소되었습니다.")